In [1]:
# test_dropdown_exclusion.ipynb

import ipywidgets as widgets
from IPython.display import display

numbers = ['1', '2', '3', '4', '5']

ask_dropdown = widgets.Dropdown(options=numbers, description="Ask:")
offer_dropdown = widgets.Dropdown(description="Offer:")

def update_offer_options(change):
    selected = change['new']
    offer_dropdown.options = [n for n in numbers if n != selected]

ask_dropdown.observe(update_offer_options, names='value')

display(ask_dropdown, offer_dropdown)

Dropdown(description='Ask:', options=('1', '2', '3', '4', '5'), value='1')

Dropdown(description='Offer:', options=(), value=None)

In [3]:
# rune_trade_summary_widget.ipynb

import json
import pandas as pd
from collections import defaultdict
from datetime import timezone
from IPython.display import display
import ipywidgets as widgets

# === Load Data ===
with open("/Users/buddy/Desktop/traderie/data/completed_trades_pc_sc_nl.json") as f:
    raw_data = json.load(f)

# === Extract Unique Runes ===
rune_set = set(r for k in raw_data for r in k.split(':'))
rune_list = sorted(rune_set)

# === Dropdowns ===
ask_dropdown = widgets.Dropdown(options=rune_list, description="Ask:")
offer_dropdown = widgets.Dropdown(description="Offer:")
output_area = widgets.Output()

def update_offer_options(change):
    selected = change['new']
    offer_dropdown.options = [r for r in rune_list if r != selected]

ask_dropdown.observe(update_offer_options, names='value')

# === Trade Summary Logic ===
def show_trade_summary(change=None):
    output_area.clear_output()
    ask = ask_dropdown.value
    offer = offer_dropdown.value
    key = f"{ask}:{offer}"
    reverse_key = f"{offer}:{ask}"
    
    with output_area:
        print(f"Trades for {ask} ⇄ {offer}\n")
        for k in [key, reverse_key]:
            if k in raw_data:
                offer_r, ask_r = k.split(':')
                print(f"{offer_r} : {ask_r}")
                for ratio_str, info in raw_data[k].get("ratios", {}).items():
                    count = info.get("count", 0)
                    last_seen = pd.to_datetime(info.get("last_seen"))
                    hours_ago = int((pd.Timestamp.now(tz=timezone.utc) - last_seen).total_seconds() // 3600)
                    print(f"  {ratio_str} — {count} trade(s), {hours_ago}h ago")
                print()
                break
        else:
            print("❌ No trade data found for this pair.")

ask_dropdown.observe(show_trade_summary, names='value')
offer_dropdown.observe(show_trade_summary, names='value')

# === Display ===
display(ask_dropdown, offer_dropdown, output_area)

Dropdown(description='Ask:', options=('Amn Rune', 'Ber Rune', 'Cham Rune', 'Dol Rune', 'El Rune', 'Eld Rune', …

Dropdown(description='Offer:', options=(), value=None)

Output()

In [5]:
# rune_summary_condensed_widget.ipynb

import json
import pandas as pd
from collections import defaultdict
from datetime import timezone
import ipywidgets as widgets
from IPython.display import display

# === Load Data ===
with open("completed_trades_pc_sc_nl.json") as f:
    raw_data = json.load(f)

# === Extract Runes ===
runes = sorted(set(r for k in raw_data for r in k.split(":")))

# === Widget ===
rune_dropdown = widgets.Dropdown(options=runes, description="Rune:")
output_area = widgets.Output()

# === Summary Display Logic ===
def show_summary(change=None):
    output_area.clear_output()
    selected_rune = rune_dropdown.value
    trade_summaries = defaultdict(list)

    for trade_key, content in raw_data.items():
        offer, ask = trade_key.split(':')
        if selected_rune in (offer, ask):
            for ratio_str, ratio_info in content.get('ratios', {}).items():
                count = ratio_info['count']
                last_seen = ratio_info['last_seen']
                time_ago = pd.Timestamp.now(tz=timezone.utc) - pd.to_datetime(last_seen)
                hours_ago = int(time_ago.total_seconds() // 3600)
                recent_str = f"{ratio_str} ({hours_ago}h ago)"
                trade_summaries[f"{offer} : {ask}"].append(recent_str)

    with output_area:
        print(f"{selected_rune} Trades\n")
        for pair, summaries in sorted(trade_summaries.items(), key=lambda x: -len(x[1])):
            print(f"{pair}")
            for line in summaries:
                print(f"  {line}")
            print()

rune_dropdown.observe(show_summary, names='value')

# === Display ===
display(rune_dropdown, output_area)
show_summary()

FileNotFoundError: [Errno 2] No such file or directory: 'completed_trades_pc_sc_nl.json'